In [16]:
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader

In [17]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [18]:
tokenizer = tiktoken.get_encoding("gpt2")
enc_text = tokenizer.encode(raw_text)

In [19]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

In [20]:
vocab_size = 50257
output_dim = 256
max_length = 4

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(max_length, output_dim)

In [21]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

token_embeddings = token_embedding_layer(inputs)
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
input_embeddings = token_embeddings + pos_embeddings

In [22]:
input_embeddings

tensor([[[-2.1976,  1.9642,  0.1532,  ..., -0.5217, -1.1677, -1.6193],
         [-1.9465, -1.1424, -0.3114,  ..., -0.7697, -0.2445,  0.7971],
         [-0.7967, -0.3491, -0.1853,  ...,  0.4970,  1.1927,  2.3502],
         [ 1.5212,  1.8974,  0.9068,  ..., -1.1109,  2.8418, -1.7065]],

        [[-1.1044,  0.4228,  0.8634,  ..., -0.7927, -0.7938,  1.2213],
         [-1.4594, -0.0253, -0.3837,  ...,  0.4147, -0.8188,  0.3145],
         [ 1.3111, -0.7813, -0.3813,  ..., -0.7569,  2.1640, -1.0096],
         [-0.5509,  0.8257,  1.2166,  ..., -0.1864, -0.0486, -1.5020]],

        [[-2.0611,  1.3720,  2.1711,  ...,  0.9265, -0.7375, -1.3977],
         [-1.7009,  1.5272, -1.2093,  ...,  0.2023,  0.9959, -2.8996],
         [ 1.0212,  0.7060, -0.7361,  ..., -0.9014,  0.9513,  0.3554],
         [ 1.2480,  1.4005,  0.6307,  ..., -2.0975,  0.1092, -0.8253]],

        ...,

        [[-2.1209,  2.3055, -0.6168,  ..., -0.9289,  1.3431, -0.3978],
         [-2.2820,  0.8748,  0.1248,  ..., -1.5296, -1.37